# Data Preprocessing

## MSc Data Science Dissertation

### Predicting Employee Performance Using Machine Learning and Explainable Artificial Intelligence (XAI)

**Student:** Funwayo Nkhata  
**Programme:** MSc Data Science  
**Supervisor:** Dr Kat Fair

### Objective

The objective of this notebook is to prepare the employee performance dataset for supervised machine-learning analysis. Building upon the Data Understanding phase, the notebook verifies data quality, removes non-informative variables, investigates predictor validity and potential target leakage, refines the predictor set, transforms categorical variables and standardises numerical features where appropriate.

A diagnostic modelling framework is used to investigate variables presenting potential leakage, temporal-ordering or conceptual-validity concerns. Sequential feature removal is complemented by individual predictor ablation and examination of the relationship between `Monthly_Salary` and `Performance_Score`. These analyses distinguish empirical model dependence from the conceptual question of whether predictors would be appropriate and available at the intended point of prediction.

To minimise preprocessing leakage, the dataset is partitioned into training and testing subsets before the categorical encoder and numerical scaler are fitted. All learned preprocessing transformations are estimated exclusively from the training data and subsequently applied to the held-out testing data.

The resulting leakage-controlled, encoded and scaled datasets provide a transparent, reproducible and methodologically defensible foundation for the comparative machine-learning analysis presented in Notebook 3.

## Import Required Libraries

The following libraries support data manipulation, feature transformation, diagnostic modelling, data partitioning and the saving of reusable preprocessing objects.

In [65]:
from pathlib import Path

import joblib
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier

print("Libraries imported successfully.")

Libraries imported successfully.


## Define File Paths

Centralising file paths improves reproducibility and reduces the risk of inconsistent references across the project.

In [66]:
DATA_PATH = Path("../Data/Employee Data.csv")
PROCESSED_DIR = Path("../Processed_Data")
MODEL_DIR = Path("../Models")

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

print("File paths configured successfully.")

File paths configured successfully.


## Load the Original Dataset

The original employee performance dataset is loaded from the project data directory. The first five observations are displayed to confirm that the data has been imported correctly.

In [67]:
df = pd.read_csv(DATA_PATH)

df.head()

,Employee_ID,Department,Gender,Age,Job_Title,Hire_Date,Years_At_Company,Education_Level,Performance_Score,Monthly_Salary,Work_Hours_Per_Week,Projects_Handled,Overtime_Hours,Sick_Days,Remote_Work_Frequency,Team_Size,Training_Hours,Promotions,Employee_Satisfaction_Score,Resigned
0,1,IT,Male,55,Specialist,03:05.6,2,High School,5,6750,33,32,22,2,0,14,66,0,2.63,False
1,2,Finance,Male,29,Developer,03:05.6,0,High School,5,7500,34,34,13,14,100,12,61,2,1.72,False
2,3,Finance,Male,55,Specialist,03:05.6,8,High School,3,5850,37,27,6,3,50,10,1,0,3.17,False
3,4,Customer Support,Female,48,Analyst,03:05.6,7,Bachelor,2,4800,52,10,28,12,100,10,0,1,1.86,False
4,5,Engineering,Female,36,Analyst,03:05.6,3,Bachelor,2,4800,38,11,29,13,100,15,9,1,1.25,False


## Verify the Dataset

The dimensions, data types and completeness of the dataset are verified before preprocessing begins. This quality-assurance step confirms that the correct source data has been loaded and that its structure remains consistent with the findings from Notebook 1.

In [68]:
print(f"Dataset shape: {df.shape}")
print(f"Number of duplicated rows: {df.duplicated().sum()}")
print(f"Total missing values: {df.isna().sum().sum()}")

df.info()

Dataset shape: (100000, 20)
Number of duplicated rows: 0
Total missing values: 0
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 20 columns):
 #   Column                       Non-Null Count   Dtype  
---  ------                       --------------   -----  
 0   Employee_ID                  100000 non-null  int64  
 1   Department                   100000 non-null  object 
 2   Gender                       100000 non-null  object 
 3   Age                          100000 non-null  int64  
 4   Job_Title                    100000 non-null  object 
 5   Hire_Date                    100000 non-null  object 
 6   Years_At_Company             100000 non-null  int64  
 7   Education_Level              100000 non-null  object 
 8   Performance_Score            100000 non-null  int64  
 9   Monthly_Salary               100000 non-null  int64  
 10  Work_Hours_Per_Week          100000 non-null  int64  
 11  Projects_Handled             100000 n

## Interpretation

The source dataset contains **100,000 observations and 20 variables**. Inspection confirms that no missing values are present and that the dataset contains numerical, categorical and Boolean variables.

Verifying the source data before transformation supports reproducibility and ensures that subsequent preprocessing decisions are applied to the expected dataset rather than to an incomplete or incorrectly loaded version.

## Remove Non-Informative Variables

The Data Understanding phase identified two variables that do not provide meaningful predictive information.

- `Employee_ID` is a unique record identifier. It distinguishes employees but does not describe characteristics that should explain performance.
- `Hire_Date` contains the same invalid value (`03:05.6`) for every observation. As the variable has no variation and does not represent a valid date, it cannot contribute to predictive modelling.

These variables are removed before the feature-selection and modelling stages.

In [69]:
df = df.drop(columns=["Employee_ID", "Hire_Date"])

print("Employee_ID and Hire_Date removed successfully.")
print("Updated shape:", df.shape)

df.head()

Employee_ID and Hire_Date removed successfully.
Updated shape: (100000, 18)


,Department,Gender,Age,Job_Title,Years_At_Company,Education_Level,Performance_Score,Monthly_Salary,Work_Hours_Per_Week,Projects_Handled,Overtime_Hours,Sick_Days,Remote_Work_Frequency,Team_Size,Training_Hours,Promotions,Employee_Satisfaction_Score,Resigned
0,IT,Male,55,Specialist,2,High School,5,6750,33,32,22,2,0,14,66,0,2.63,False
1,Finance,Male,29,Developer,0,High School,5,7500,34,34,13,14,100,12,61,2,1.72,False
2,Finance,Male,55,Specialist,8,High School,3,5850,37,27,6,3,50,10,1,0,3.17,False
3,Customer Support,Female,48,Analyst,7,Bachelor,2,4800,52,10,28,12,100,10,0,1,1.86,False
4,Engineering,Female,36,Analyst,3,Bachelor,2,4800,38,11,29,13,100,15,9,1,1.25,False


## Verify Variable Removal

The revised dataset is inspected to confirm that the non-informative variables have been removed successfully.

In [70]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 18 columns):
 #   Column                       Non-Null Count   Dtype  
---  ------                       --------------   -----  
 0   Department                   100000 non-null  object 
 1   Gender                       100000 non-null  object 
 2   Age                          100000 non-null  int64  
 3   Job_Title                    100000 non-null  object 
 4   Years_At_Company             100000 non-null  int64  
 5   Education_Level              100000 non-null  object 
 6   Performance_Score            100000 non-null  int64  
 7   Monthly_Salary               100000 non-null  int64  
 8   Work_Hours_Per_Week          100000 non-null  int64  
 9   Projects_Handled             100000 non-null  int64  
 10  Overtime_Hours               100000 non-null  int64  
 11  Sick_Days                    100000 non-null  int64  
 12  Remote_Work_Frequency        100000 non-null  int64  
 13  

## Interpretation

Inspection confirms that `Employee_ID` and `Hire_Date` have been removed. The dataset now contains 18 variables, including the target variable and the features that require further methodological evaluation.

At this stage, variables associated with potential leakage, temporal-ordering or conceptual-validity concerns remain temporarily available so that their empirical effect on model performance can be investigated before the final predictor set is determined.

## Investigation of Predictor Validity and Potential Target Leakage

During initial modelling, the predictor set included `Monthly_Salary`, `Promotions` and `Resigned`. A Decision Tree trained using this feature set achieved perfect held-out accuracy, an unexpected result that warranted investigation before proceeding to comparative model development. Perfect predictive performance does not itself demonstrate target leakage; it may instead reflect an unusually strong relationship embedded within the synthetic data-generating process. The three variables were therefore examined both empirically and conceptually.

`Monthly_Salary` presents a particular concern because remuneration may encode information associated with organisational grade, responsibility or prior performance outcomes. `Promotions` presents a temporal-ordering concern because promotion decisions may occur after, or be informed by, employee performance assessment. `Resigned` is temporally and conceptually ambiguous because resignation may occur after the assessment period and may reflect factors unrelated to performance.

The investigation uses two complementary diagnostic approaches. First, a staged feature-removal experiment examines how predictive performance changes as the three variables are progressively excluded. Second, an individual predictor ablation analysis removes each variable independently and in combination, addressing the limitation that sequential removal cannot isolate the contribution of variables removed later in the sequence. The relationship between `Monthly_Salary` and `Performance_Score` is subsequently examined to investigate the empirical structure underlying the observed model performance.

All diagnostic scenarios use the same stratified training and testing observations, Decision Tree configuration and random state. Consequently, differences in held-out accuracy can be attributed to changes in predictor composition rather than changes in the data partition or model specification. These experiments assess model dependence on the candidate predictors; whether a variable constitutes target leakage additionally depends on its temporal availability and conceptual relationship with the prediction target.

### Define the Target and Diagnostic Predictor Sets

The target variable, `Performance_Score`, is separated from the predictors. Four predictor scenarios are then defined to examine how Decision Tree performance changes as variables associated with potential leakage, temporal ambiguity or conceptual-validity concerns are progressively removed.

The staged scenarios are:

1. **Initial feature set:** `Monthly_Salary`, `Promotions` and `Resigned` are retained.
2. **Monthly_Salary removed:** `Promotions` and `Resigned` are retained.
3. **Monthly_Salary and Promotions removed:** only `Resigned` remains among the three candidate variables.
4. **Refined feature set:** `Monthly_Salary`, `Promotions` and `Resigned` are all excluded.

The refined feature set represents the predictors proposed for subsequent comparative modelling. The staged analysis is diagnostic rather than a model-selection procedure and is used to evaluate how the composition of the predictor set affects held-out predictive performance.

In [71]:
TARGET = "Performance_Score"

y = df[TARGET].copy()

# Scenario A: all candidate variables retained
X_initial = df.drop(columns=[TARGET]).copy()

# Scenario B: Monthly_Salary removed
X_without_salary = df.drop(
    columns=[TARGET, "Monthly_Salary"]
).copy()

# Scenario C: Monthly_Salary and Promotions removed
X_without_salary_promotions = df.drop(
    columns=[
        TARGET,
        "Monthly_Salary",
        "Promotions"
    ]
).copy()

# Scenario D: final refined feature set
X_refined = df.drop(
    columns=[
        TARGET,
        "Monthly_Salary",
        "Promotions",
        "Resigned"
    ]
).copy()

print("Initial feature-set shape:", X_initial.shape)
print("Without salary shape:", X_without_salary.shape)
print(
    "Without salary and promotions shape:",
    X_without_salary_promotions.shape
)
print("Refined feature-set shape:", X_refined.shape)
print("Target shape:", y.shape)

Initial feature-set shape: (100000, 17)
Without salary shape: (100000, 16)
Without salary and promotions shape: (100000, 15)
Refined feature-set shape: (100000, 14)
Target shape: (100000,)


### Create a Consistent Diagnostic Train/Test Partition

To ensure direct comparability across the diagnostic scenarios, the same training and testing observations are used throughout the investigation.

An 80:20 stratified partition is created using `random_state=42`. Stratification preserves the approximately equal distribution of the five `Performance_Score` categories across the training and testing subsets, while the fixed random state supports reproducibility.

Only predictor composition changes between diagnostic scenarios. The training observations, testing observations, Decision Tree configuration and random state remain constant. This controlled design reduces the possibility that differences in observed accuracy arise from different data partitions rather than from the variables being investigated.

In [72]:
# Create one consistent stratified split for all diagnostic scenarios
train_indices, test_indices = train_test_split(
    df.index,
    test_size=0.20,
    random_state=42,
    stratify=y
)

y_train_diagnostic = y.loc[train_indices].copy()
y_test_diagnostic = y.loc[test_indices].copy()

print("Diagnostic training observations:", len(train_indices))
print("Diagnostic testing observations:", len(test_indices))

print("\nTraining target distribution:")
print(
    y_train_diagnostic.value_counts(normalize=True)
    .sort_index()
    .round(4)
)

print("\nTesting target distribution:")
print(
    y_test_diagnostic.value_counts(normalize=True)
    .sort_index()
    .round(4)
)

Diagnostic training observations: 80000
Diagnostic testing observations: 20000

Training target distribution:
Performance_Score
1    0.2012
2    0.2001
3    0.2000
4    0.1994
5    0.1993
Name: proportion, dtype: float64

Testing target distribution:
Performance_Score
1    0.2012
2    0.2002
3    0.2000
4    0.1994
5    0.1992
Name: proportion, dtype: float64


### Define the Diagnostic Modelling Pipeline

Each diagnostic scenario may contain numerical, categorical and Boolean predictors. A reusable preprocessing and modelling pipeline is therefore defined to ensure that each scenario is treated consistently.

Nominal categorical variables are transformed using One-Hot Encoding, with previously unseen categories handled using `handle_unknown="ignore"`. Numerical and Boolean predictors are passed through without standardisation because Decision Trees partition observations using feature thresholds and are not sensitive to differences in measurement scale.

For each scenario, the encoder and Decision Tree are fitted exclusively using the corresponding training observations and are then evaluated on the held-out testing observations. This prevents information from the testing subset from influencing the fitted preprocessing transformation.

The Decision Tree is used here as a diagnostic instrument rather than as the final selected predictive model. Its purpose is to provide a consistent test of how changes in predictor composition affect held-out predictive performance. Comparative model development and evaluation are undertaken separately in Notebook 3.

In [73]:
def build_diagnostic_pipeline(X_scenario):
    """
    Construct a preprocessing and Decision Tree pipeline
    for a specified diagnostic predictor dataset.
    """

    diagnostic_categorical_features = (
        X_scenario.select_dtypes(
            include=["object", "category"]
        ).columns.tolist()
    )

    diagnostic_numeric_features = (
        X_scenario.select_dtypes(
            include=["number", "bool"]
        ).columns.tolist()
    )

    diagnostic_preprocessor = ColumnTransformer(
        transformers=[
            (
                "categorical",
                OneHotEncoder(
                    handle_unknown="ignore",
                    sparse_output=False
                ),
                diagnostic_categorical_features
            ),
            (
                "numerical",
                "passthrough",
                diagnostic_numeric_features
            )
        ],
        remainder="drop"
    )

    diagnostic_pipeline = Pipeline(
        steps=[
            ("preprocessor", diagnostic_preprocessor),
            (
                "classifier",
                DecisionTreeClassifier(
                    random_state=42
                )
            )
        ]
    )

    return diagnostic_pipeline


print("Diagnostic pipeline function created successfully.")

Diagnostic pipeline function created successfully.


In [74]:
diagnostic_scenarios = {
    "Initial feature set": X_initial,
    "Monthly_Salary removed": X_without_salary,
    "Monthly_Salary and Promotions removed":
        X_without_salary_promotions,
    "Refined feature set": X_refined
}

diagnostic_results = []

for scenario_name, X_scenario in diagnostic_scenarios.items():

    pipeline = build_diagnostic_pipeline(X_scenario)

    pipeline.fit(
        X_scenario.loc[train_indices],
        y_train_diagnostic
    )

    predictions = pipeline.predict(
        X_scenario.loc[test_indices]
    )

    accuracy = accuracy_score(
        y_test_diagnostic,
        predictions
    )

    diagnostic_results.append(
        {
            "Scenario": scenario_name,
            "Monthly_Salary": (
                "Included"
                if "Monthly_Salary" in X_scenario.columns
                else "Excluded"
            ),
            "Promotions": (
                "Included"
                if "Promotions" in X_scenario.columns
                else "Excluded"
            ),
            "Resigned": (
                "Included"
                if "Resigned" in X_scenario.columns
                else "Excluded"
            ),
            "Decision_Tree_Accuracy": accuracy
        }
    )

leakage_comparison = pd.DataFrame(diagnostic_results)

leakage_comparison["Decision_Tree_Accuracy"] = (
    leakage_comparison["Decision_Tree_Accuracy"].round(4)
)

leakage_comparison

,Scenario,Monthly_Salary,Promotions,Resigned,Decision_Tree_Accuracy
0,Initial feature set,Included,Included,Included,1.0000
1,Monthly_Salary removed,Excluded,Included,Included,0.1995
2,Monthly_Salary and Promotions removed,Excluded,Excluded,Included,0.1986
3,Refined feature set,Excluded,Excluded,Excluded,0.1986


### Individual Predictor Ablation Analysis

The staged feature-removal experiment evaluates the cumulative effect of progressively removing potentially problematic predictors. However, a sequential procedure has an important limitation: once one predictor has been removed, subsequent stages cannot establish whether variables removed later would have affected performance independently.

To address this limitation, an individual predictor ablation analysis is conducted. `Monthly_Salary`, `Promotions` and `Resigned` are removed individually and in combination while maintaining the same training-test partition, preprocessing procedure and Decision Tree configuration.

This sensitivity analysis allows the effect of removal order to be distinguished from the independent predictive contribution of each candidate variable. Because the dataset is synthetic and the temporal provenance of these variables is uncertain, reductions in predictive performance are interpreted as evidence of model dependence rather than, by themselves, proof of target leakage.


In [75]:
ablation_scenarios = {
    "All predictors": [],
    "Remove Monthly_Salary only": ["Monthly_Salary"],
    "Remove Promotions only": ["Promotions"],
    "Remove Resigned only": ["Resigned"],
    "Remove Monthly_Salary + Promotions": [
        "Monthly_Salary", "Promotions"
    ],
    "Remove Monthly_Salary + Resigned": [
        "Monthly_Salary", "Resigned"
    ],
    "Remove Promotions + Resigned": [
        "Promotions", "Resigned"
    ],
    "Remove all three": [
        "Monthly_Salary", "Promotions", "Resigned"
    ]
}

ablation_results = []

for scenario_name, columns_to_remove in ablation_scenarios.items():

    X_scenario = df.drop(
        columns=[TARGET] + columns_to_remove
    ).copy()

    pipeline = build_diagnostic_pipeline(X_scenario)

    pipeline.fit(
        X_scenario.loc[train_indices],
        y_train_diagnostic
    )

    predictions = pipeline.predict(
        X_scenario.loc[test_indices]
    )

    accuracy = accuracy_score(
        y_test_diagnostic,
        predictions
    )

    ablation_results.append({
        "Scenario": scenario_name,
        "Accuracy": accuracy
    })

ablation_comparison = pd.DataFrame(ablation_results)

ablation_comparison["Accuracy"] = (
    ablation_comparison["Accuracy"].round(4)
)

ablation_comparison

,Scenario,Accuracy
0,All predictors,1.0000
1,Remove Monthly_Salary only,0.1995
2,Remove Promotions only,1.0000
3,Remove Resigned only,1.0000
4,Remove Monthly_Salary + Promotions,0.1986
5,Remove Monthly_Salary + Resigned,0.2002
6,Remove Promotions + Resigned,1.0000
7,Remove all three,0.1986


#### Interpretation of Individual Predictor Ablation

The individual ablation analysis provides stronger evidence about the source of the perfect predictive performance observed in the initial model. Removing `Monthly_Salary` alone reduced held-out accuracy from 1.0000 to 0.1995. In contrast, removing either `Promotions` or `Resigned` individually left accuracy unchanged at 1.0000. Removing both `Promotions` and `Resigned` simultaneously while retaining `Monthly_Salary` also preserved perfect accuracy.

These results demonstrate that the Decision Tree's perfect classification is overwhelmingly dependent on information contained in `Monthly_Salary`, rather than on `Promotions` or `Resigned`.

Once `Monthly_Salary` was excluded, further removal of the other candidate variables produced only small fluctuations around 20% accuracy. Removing salary and promotions resulted in an accuracy of 0.1986, removing salary and resignation produced 0.2002, and removing all three variables produced 0.1986. Given the approximately balanced five-class target, these differences are small and provide little evidence that either `Promotions` or `Resigned` independently contributes substantial predictive information.

Importantly, the ablation experiment establishes predictive dependence rather than causation or target leakage itself. The strong contribution of `Monthly_Salary` demonstrates that the fitted Decision Tree can exploit salary information to recover the target with exceptional accuracy. Determining why this relationship exists requires examination of the underlying salary-performance structure and consideration of the synthetic data-generation process.

### Diagnostic Examination of Monthly Salary and Performance Score

The substantial reduction in predictive accuracy following removal of `Monthly_Salary` warrants further investigation of the relationship between this predictor and the target.

Descriptive statistics are first calculated for `Monthly_Salary` within each `Performance_Score` category to examine differences in salary distributions. The number of unique salary values within each target category is then assessed, followed by cross-tabulation of salary values against performance categories to identify whether particular salary values are systematically associated with the target.

Because the dataset is synthetic, a strong relationship cannot automatically be interpreted as conventional temporal target leakage. It may instead reflect the rules or dependencies embedded within the synthetic data-generation process. The purpose of this diagnostic analysis is therefore to establish why salary provides such strong predictive information, rather than to infer a causal relationship between remuneration and employee performance.

In [76]:
salary_by_performance = (
    df.groupby("Performance_Score")["Monthly_Salary"]
      .agg(["count", "mean", "median", "min", "max", "std"])
)

salary_by_performance

,count,mean,median,min,max,std
Performance_Score,,,,,,
1,20120,5422.229125,5500.0,3850,6600,991.495988
2,20013,5900.284815,6000.0,4200,7200,1087.571943
3,19999,6409.515476,6500.0,4550,7800,1175.934200
4,19940,6897.632899,7000.0,4900,8400,1265.868795
5,19928,7397.669109,7500.0,5250,9000,1349.984698


In [77]:
salary_unique_values = (
    df.groupby("Performance_Score")["Monthly_Salary"]
      .nunique()
)

salary_unique_values

Performance_Score
1    6
2    6
3    6
4    6
5    6
Name: Monthly_Salary, dtype: int64

In [78]:
salary_performance_table = pd.crosstab(
    df["Monthly_Salary"],
    df["Performance_Score"]
)

salary_performance_table

Performance_Score,1,2,3,4,5
Monthly_Salary,,,,,
3850,2855,0,0,0,0
4200,0,2927,0,0,0
4400,2858,0,0,0,0
4550,0,0,2858,0,0
4800,0,2891,0,0,0
4900,0,0,0,2851,0
4950,2889,0,0,0,0
5200,0,0,2859,0,0
5250,0,0,0,0,2794


#### Interpretation of the Salary–Performance Relationship

The salary diagnostics provide a plausible empirical explanation for the Decision Tree's dependence on `Monthly_Salary`. Both mean and median salary increase progressively across the five performance categories. Mean monthly salary rises from approximately 5,422 for Performance Score 1 to 7,398 for Performance Score 5, while median salary increases from 5,500 to 7,500.

More importantly, the cross-tabulation reveals a highly structured relationship between salary values and performance categories. Many salary values occur exclusively within a single `Performance_Score` category, meaning that salary frequently provides sufficient information to distinguish the target class. Although some salary values overlap between categories, the overall structure provides a clear mechanism through which the Decision Tree can achieve perfect held-out classification when `Monthly_Salary` is retained alongside the remaining predictors.

This finding should not be interpreted as evidence that salary causes employee performance. Nor, because the dataset is synthetic and the temporal provenance of salary is not established, does the relationship alone prove conventional target leakage. Instead, the evidence indicates that `Monthly_Salary` functions as an exceptionally strong proxy for `Performance_Score` within this dataset and may reflect dependencies embedded in the synthetic data-generation process.

Retaining such a predictor would therefore risk producing an unrealistic representation of prospective employee-performance prediction. Its exclusion is justified on the basis of predictive validity and prospective applicability: the study seeks to evaluate whether performance can be predicted from conceptually appropriate antecedent characteristics rather than from a variable that contains unusually direct information about the target.

### Overall Interpretation of the Diagnostic Investigation

The combined diagnostic analyses demonstrate that the initially perfect Decision Tree performance was overwhelmingly dependent on `Monthly_Salary`. In the staged experiment, removing salary reduced held-out accuracy from 1.0000 to 0.1995. The individual ablation analysis strengthens this conclusion: removing either `Promotions` or `Resigned` while retaining salary preserved perfect accuracy, as did removing both variables together. By contrast, every scenario excluding `Monthly_Salary` produced accuracy close to 20%.

The salary-distribution analysis provides a plausible empirical explanation for this result. `Monthly_Salary` is highly structured according to `Performance_Score`, with progressively increasing salary distributions across performance categories and many salary values associated with individual target classes. The Decision Tree can therefore exploit salary as an exceptionally informative proxy for the target.

The evidence for `Promotions` and `Resigned` is different. Neither variable independently explains the perfect predictive performance, and once salary is removed their inclusion or exclusion produces only small changes around 20% accuracy. Their removal is therefore not justified by evidence that they independently generate substantial predictive inflation. Instead, they are excluded because their temporal relationship with the intended prediction point is uncertain: promotion and resignation may occur after, or partly in response to, the performance period being predicted.

These distinctions are methodologically important. High predictive importance does not by itself establish target leakage, while weak predictive importance does not establish that a variable is conceptually appropriate for prospective prediction. Predictor validity depends both on empirical model behaviour and on whether the information represented by a predictor would legitimately be available at the intended prediction point.

Overall, the diagnostic investigation supports a conservative refined feature set that excludes `Monthly_Salary`, `Promotions` and `Resigned`. The resulting reduction in apparent predictive performance is not treated as a modelling failure. Rather, it enables a more credible evaluation of whether employee performance can be predicted from the remaining antecedent employee and organisational characteristics.

## Final Feature Selection and Leakage-Risk Mitigation

Following the staged feature-removal experiment, individual predictor ablation and salary-performance diagnostic analysis, `Monthly_Salary`, `Promotions` and `Resigned` are excluded from the final predictor set.

`Monthly_Salary` is excluded because the empirical analysis demonstrates exceptional model dependence on this variable. Removing salary alone reduces Decision Tree accuracy from 1.0000 to approximately 0.20, while retaining salary preserves perfect accuracy even when `Promotions` and `Resigned` are removed. The salary diagnostics further demonstrate a highly structured relationship between salary values and performance categories. Within this synthetic dataset, salary therefore acts as an unusually strong proxy for the target and would undermine a realistic assessment of prospective predictive capability.

`Promotions` and `Resigned` are excluded for different reasons. The ablation analysis provides little evidence that either variable independently drives predictive performance. Their exclusion is instead based primarily on temporal and conceptual validity. Promotion may represent an outcome of previous performance, while resignation may occur after the assessment period and may reflect factors unrelated to performance. Without reliable temporal metadata establishing that these variables are known before the intended prediction point, their inclusion would weaken the prospective interpretation of the model.

The final feature-selection strategy therefore combines empirical evidence with temporal and conceptual reasoning rather than defining target leakage solely according to changes in predictive accuracy. This produces a more defensible predictor set for evaluating how well employee performance can be predicted using information plausibly available before the target outcome is known.

## Define the Final Predictor and Target Datasets

The refined predictor set established through the diagnostic investigation is adopted as the final feature set for comparative modelling. `Monthly_Salary`, `Promotions` and `Resigned` are excluded, while `Performance_Score` remains the multiclass target variable.

The purpose of this refined dataset is to evaluate predictive performance using predictors considered more defensible for prospective application, rather than maximising accuracy through variables that present substantial proxy, temporal-ordering or conceptual-validity concerns.

In [79]:
# Adopt the leakage-controlled predictor set for final modelling
X = X_refined.copy()

# Retain the previously defined target
y = df[TARGET].copy()

print("Final predictor shape:", X.shape)
print("Target shape:", y.shape)

Final predictor shape: (100000, 14)
Target shape: (100000,)


### Interpretation

The final predictor dataset contains 14 explanatory variables, while the target dataset contains the `Performance_Score` for each of the 100,000 observations.

The refined predictor set excludes the non-informative `Employee_ID` and `Hire_Date` variables together with `Monthly_Salary`, `Promotions` and `Resigned`, which presented proxy, temporal-ordering or conceptual-validity concerns.

The final predictor set therefore provides a more defensible basis for prospective employee-performance modelling. Importantly, the feature-selection process prioritises predictor validity over maximising apparent predictive accuracy. Subsequent model performance should therefore be interpreted as an estimate of the predictive information contained within the refined antecedent feature set rather than as a direct comparison with the exceptionally high performance obtained when `Monthly_Salary` was retained.

## Identify Predictor Variable Types

Different data types require different preprocessing treatments. Numerical features may require standardisation for scale-sensitive algorithms, while nominal categorical features require numerical encoding.

Identifying feature types before transformation ensures that each variable receives an appropriate and reproducible preprocessing treatment.

In [80]:
numerical_features = X.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

categorical_features = X.select_dtypes(
    include=["object", "category"]
).columns.tolist()

boolean_features = X.select_dtypes(
    include=["bool"]
).columns.tolist()

print("Numerical Features:")
print(numerical_features)

print("\nCategorical Features:")
print(categorical_features)

print("\nBoolean Features:")
print(boolean_features)

Numerical Features:
['Age', 'Years_At_Company', 'Work_Hours_Per_Week', 'Projects_Handled', 'Overtime_Hours', 'Sick_Days', 'Remote_Work_Frequency', 'Team_Size', 'Training_Hours', 'Employee_Satisfaction_Score']

Categorical Features:
['Department', 'Gender', 'Job_Title', 'Education_Level']

Boolean Features:
[]


## Interpretation

The final predictor dataset contains **10 numerical variables** and **4 categorical variables**. No Boolean predictors remain because `Resigned` was removed during feature selection.

The numerical variables will be retained in their original form for tree-based algorithms and standardised for scale-sensitive algorithms. The categorical variables will be transformed using One-Hot Encoding.

## Partition the Raw Predictor Dataset

The refined predictor dataset is divided into training and testing subsets before encoding and scaling.

An 80:20 stratified split is used, with `random_state=42` ensuring reproducibility. The same indices used during the diagnostic experiment are retained so that all subsequent comparisons are based on an identical data partition.

Performing the split before fitting preprocessing transformations is methodologically important. It prevents the categorical encoder and numerical scaler from learning information from the testing data.

In [81]:
X_train_raw = X.loc[train_indices].copy()
X_test_raw = X.loc[test_indices].copy()

y_train = y.loc[train_indices].copy()
y_test = y.loc[test_indices].copy()

print("X_train_raw:", X_train_raw.shape)
print("X_test_raw:", X_test_raw.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

X_train_raw: (80000, 14)
X_test_raw: (20000, 14)
y_train: (80000,)
y_test: (20000,)


## Verify Class Distribution

The distribution of the target variable is examined across the full, training and testing datasets to confirm that stratification has preserved the balance of the five performance categories.

In [82]:
class_distribution = pd.DataFrame(
    {
        "Full_Count": y.value_counts().sort_index(),
        "Train_Count": y_train.value_counts().sort_index(),
        "Test_Count": y_test.value_counts().sort_index(),
        "Full_Percentage": (
            y.value_counts(normalize=True).sort_index() * 100
        ).round(2),
        "Train_Percentage": (
            y_train.value_counts(normalize=True).sort_index() * 100
        ).round(2),
        "Test_Percentage": (
            y_test.value_counts(normalize=True).sort_index() * 100
        ).round(2)
    }
)

class_distribution

,Full_Count,Train_Count,Test_Count,Full_Percentage,Train_Percentage,Test_Percentage
Performance_Score,,,,,,
1,20120,16096,4024,20.12,20.12,20.12
2,20013,16010,4003,20.01,20.01,20.02
3,19999,15999,4000,20.00,20.00,20.00
4,19940,15952,3988,19.94,19.94,19.94
5,19928,15943,3985,19.93,19.93,19.92


### Interpretation

The stratified partition preserves the approximately equal distribution of the five `Performance_Score` categories across the full, training and testing datasets. Each class represents approximately 20% of observations, with only minor differences arising from the original class frequencies.

This confirms that the training and held-out testing subsets remain representative of the target distribution and reduces the risk that comparative model evaluation is distorted by disproportionate representation of any performance category.

The same partition is retained across all models in Notebook 3, supporting fair comparison because differences in held-out performance cannot be attributed to different train-test samples.

## Encode Categorical Variables

One-Hot Encoding transforms each nominal category into a binary indicator feature without introducing an artificial numerical hierarchy.

The encoder is fitted exclusively on the training data. The fitted transformation is then applied to the testing data, with previously unseen test categories handled safely through `handle_unknown="ignore"`.

The parameter `drop="first"` removes one reference category from each categorical variable. This reduces redundant dummy variables and is particularly useful for linear algorithms that may be affected by exact multicollinearity.

In [83]:
encoder = OneHotEncoder(
    drop="first",
    handle_unknown="ignore",
    sparse_output=False,
    dtype=np.int64
)

encoder.fit(X_train_raw[categorical_features])

encoded_feature_names = encoder.get_feature_names_out(
    categorical_features
)

X_train_categorical = pd.DataFrame(
    encoder.transform(X_train_raw[categorical_features]),
    columns=encoded_feature_names,
    index=X_train_raw.index
)

X_test_categorical = pd.DataFrame(
    encoder.transform(X_test_raw[categorical_features]),
    columns=encoded_feature_names,
    index=X_test_raw.index
)

print("Categorical encoding completed.")
print("Encoded categorical features:", len(encoded_feature_names))

Categorical encoding completed.
Encoded categorical features: 19


## Create Encoded Training and Testing Datasets

The unchanged numerical variables are combined with the encoded categorical indicators. These encoded datasets are suitable for tree-based algorithms such as Decision Tree, Random Forest and XGBoost.

In [84]:
X_train_encoded = pd.concat(
    [
        X_train_raw[numerical_features],
        X_train_categorical
    ],
    axis=1
)

X_test_encoded = pd.concat(
    [
        X_test_raw[numerical_features],
        X_test_categorical
    ],
    axis=1
)

print("X_train_encoded:", X_train_encoded.shape)
print("X_test_encoded:", X_test_encoded.shape)

X_train_encoded.head()

X_train_encoded: (80000, 29)
X_test_encoded: (20000, 29)


,Age,Years_At_Company,Work_Hours_Per_Week,Projects_Handled,Overtime_Hours,Sick_Days,Remote_Work_Frequency,Team_Size,Training_Hours,Employee_Satisfaction_Score,...,Gender_Other,Job_Title_Consultant,Job_Title_Developer,Job_Title_Engineer,Job_Title_Manager,Job_Title_Specialist,Job_Title_Technician,Education_Level_High School,Education_Level_Master,Education_Level_PhD
71599,49,2,60,3,27,9,0,7,92,2.07,...,0,0,0,0,0,0,1,0,1,0
64079,45,3,51,36,4,9,0,3,80,2.04,...,0,0,0,0,0,1,0,0,0,0
82730,54,6,35,39,15,12,100,16,5,1.14,...,0,0,0,0,1,0,0,0,0,0
1640,57,2,31,49,20,9,75,5,55,4.74,...,0,0,1,0,0,0,0,1,0,0
36596,28,6,48,17,2,3,25,14,4,2.38,...,0,0,0,0,0,0,0,0,1,0


### Interpretation

The four nominal categorical predictors have been transformed into binary indicator variables using One-Hot Encoding, increasing the predictor space from 14 original variables to 29 machine-learning-ready features.

The encoder was fitted exclusively on the training partition and subsequently applied to the held-out testing partition. This preserves the separation between model development and evaluation and prevents information about the testing data from influencing the learned categorical transformation.

All resulting predictors are numerical. The encoded datasets retain the original scale of the continuous and discrete numerical variables and are therefore suitable for tree-based algorithms such as Decision Tree, Random Forest and XGBoost, which do not require feature standardisation.

## Reconstruct the Full Encoded Dataset

For completeness and reproducibility, the encoded training and testing partitions are recombined and restored to their original row order. Because the encoder was fitted exclusively on the training partition before being applied to the testing partition, the reconstructed dataset remains consistent with the train-fitted encoding scheme.

The full encoded dataset is retained as a reusable project artefact. However, predictive model evaluation remains based on the predefined held-out testing partition, and the principal SHAP explainability analysis in Notebook 4 is conducted on held-out test observations after model fitting on the training data. This maintains methodological separation between model fitting and post-hoc evaluation.

In [85]:
X_encoded = pd.concat(
    [X_train_encoded, X_test_encoded],
    axis=0
).sort_index()

print("Full encoded dataset:", X_encoded.shape)

Full encoded dataset: (100000, 29)


## Create Scaled Feature Sets

Standardisation is applied only to the original numerical variables. One-hot encoded binary features remain unchanged.

The `StandardScaler` is fitted exclusively on the numerical variables within the training dataset. The fitted scaler is subsequently applied to both the training and testing numerical features. This prevents information about the distribution of the test data from influencing the preprocessing stage.

Scaled datasets are appropriate for algorithms whose optimisation procedures are sensitive to feature magnitude, including Logistic Regression and Support Vector Machines.

In [86]:
scaler = StandardScaler()

X_train_scaled = X_train_encoded.copy()
X_test_scaled = X_test_encoded.copy()

X_train_scaled[numerical_features] = scaler.fit_transform(
    X_train_encoded[numerical_features]
)

X_test_scaled[numerical_features] = scaler.transform(
    X_test_encoded[numerical_features]
)

print("X_train_scaled:", X_train_scaled.shape)
print("X_test_scaled:", X_test_scaled.shape)

X_train_scaled: (80000, 29)
X_test_scaled: (20000, 29)


## Verify Standardisation

The descriptive statistics of the scaled numerical training variables are inspected to confirm that the features are approximately centred on zero with unit variance.

In [87]:
X_train_scaled[numerical_features].describe().round(4)

,Age,Years_At_Company,Work_Hours_Per_Week,Projects_Handled,Overtime_Hours,Sick_Days,Remote_Work_Frequency,Team_Size,Training_Hours,Employee_Satisfaction_Score
count,80000.0000,80000.0000,80000.0000,80000.0000,80000.0000,80000.0000,80000.0000,80000.0000,80000.0000,80000.0000
mean,0.0000,0.0000,-0.0000,-0.0000,0.0000,-0.0000,0.0000,0.0000,0.0000,0.0000
std,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000
min,-1.6946,-1.5642,-1.6721,-1.6854,-1.6762,-1.6157,-1.4185,-1.6405,-1.7145,-1.7342
25%,-0.8931,-0.8667,-0.8895,-0.8571,-0.8680,-0.9230,-0.7107,-0.9120,-0.8486,-0.8571
50%,-0.0025,-0.1692,0.0048,-0.0288,0.0557,0.0006,-0.0029,-0.0014,-0.0173,-0.0061
75%,0.8880,0.8770,0.8991,0.8686,0.8640,0.9243,0.7048,0.9092,0.8833,0.8710
max,1.6896,1.9232,1.6816,1.6970,1.6722,1.6170,1.4126,1.6378,1.7146,1.7394


### Interpretation

The transformed numerical training features have means approximately equal to zero and standard deviations approximately equal to one, confirming that standardisation has been applied successfully.

The `StandardScaler` was fitted exclusively on the numerical predictors within the training partition and subsequently applied to the held-out testing data. Consequently, information about the distribution of the testing observations did not influence the estimated scaling parameters.

Separate feature representations are retained according to algorithmic requirements. Tree-based models use the encoded but unscaled datasets, while Logistic Regression and Linear Support Vector Machine use the standardised datasets. This avoids unnecessary transformation while ensuring that scale-sensitive algorithms are not disproportionately influenced by predictors with larger numerical magnitudes.

## Reconstruct the Full Scaled Dataset

The scaled training and testing partitions are recombined in their original row order. This provides a complete scaled feature dataset while preserving the train-fitted transformation.

In [88]:
X_scaled = pd.concat(
    [X_train_scaled, X_test_scaled],
    axis=0
).sort_index()

print("Full scaled dataset:", X_scaled.shape)

Full scaled dataset: (100000, 29)


## Final Data Quality and Alignment Checks

Before saving the processed datasets, several validation checks are performed:

- training and testing encoded datasets must contain identical columns;
- scaled and encoded datasets must have matching dimensions;
- predictor and target indices must align;
- all transformed features must be numerical;
- no missing values should have been introduced.

In [89]:
# Confirm that training and testing datasets have identical columns
assert X_train_encoded.columns.equals(
    X_test_encoded.columns
)

assert X_train_scaled.columns.equals(
    X_test_scaled.columns
)

# Confirm that encoded and scaled versions have matching dimensions
assert X_train_encoded.shape == X_train_scaled.shape
assert X_test_encoded.shape == X_test_scaled.shape

# Confirm predictor and target alignment
assert X_train_encoded.index.equals(y_train.index)
assert X_test_encoded.index.equals(y_test.index)

# Confirm complete-dataset dimensions
assert X_encoded.shape[0] == y.shape[0]
assert X_scaled.shape[0] == y.shape[0]

# Confirm no missing values were introduced
assert X_train_encoded.isna().sum().sum() == 0
assert X_test_encoded.isna().sum().sum() == 0
assert X_train_scaled.isna().sum().sum() == 0
assert X_test_scaled.isna().sum().sum() == 0

# Confirm all processed predictors are numerical
assert all(
    pd.api.types.is_numeric_dtype(dtype)
    for dtype in X_train_encoded.dtypes
)

# Confirm leakage-controlled variables are absent
excluded_predictors = {
    "Monthly_Salary",
    "Promotions",
    "Resigned",
    "Employee_ID",
    "Hire_Date"
}

assert excluded_predictors.isdisjoint(X.columns)
assert excluded_predictors.isdisjoint(X_train_encoded.columns)
assert excluded_predictors.isdisjoint(X_test_encoded.columns)

# Confirm expected dataset sizes
assert X.shape == (100000, 14)
assert X_train_encoded.shape[0] == 80000
assert X_test_encoded.shape[0] == 20000

print("All final quality-assurance checks passed.")
print("Confirmed that excluded predictors are absent.")

All final quality-assurance checks passed.
Confirmed that excluded predictors are absent.


## Processed Dataset Summary

In [90]:
processed_summary = pd.DataFrame(
    {
        "Dataset": [
            "X_encoded",
            "X_scaled",
            "X_train_encoded",
            "X_test_encoded",
            "X_train_scaled",
            "X_test_scaled",
            "y",
            "y_train",
            "y_test"
        ],
        "Rows": [
            X_encoded.shape[0],
            X_scaled.shape[0],
            X_train_encoded.shape[0],
            X_test_encoded.shape[0],
            X_train_scaled.shape[0],
            X_test_scaled.shape[0],
            y.shape[0],
            y_train.shape[0],
            y_test.shape[0]
        ],
        "Columns": [
            X_encoded.shape[1],
            X_scaled.shape[1],
            X_train_encoded.shape[1],
            X_test_encoded.shape[1],
            X_train_scaled.shape[1],
            X_test_scaled.shape[1],
            1,
            1,
            1
        ]
    }
)

processed_summary

,Dataset,Rows,Columns
0,X_encoded,100000,29
1,X_scaled,100000,29
2,X_train_encoded,80000,29
3,X_test_encoded,20000,29
4,X_train_scaled,80000,29
5,X_test_scaled,20000,29
6,y,100000,1
7,y_train,80000,1
8,y_test,20000,1


## Save Processed Datasets and Preprocessing Objects

The processed datasets are saved to ensure that Notebook 3 and Notebook 4 use consistent inputs.

The fitted encoder and scaler are also saved. Preserving these objects supports reproducibility and enables identical transformations to be applied to future observations without refitting the preprocessing operations.

In [91]:
# Save complete predictor and target datasets
X_encoded.to_csv(
    PROCESSED_DIR / "X_encoded.csv",
    index=False
)

X_scaled.to_csv(
    PROCESSED_DIR / "X_scaled.csv",
    index=False
)

y.rename(TARGET).to_csv(
    PROCESSED_DIR / "y.csv",
    index=False
)

# Save training and testing predictor datasets
X_train_encoded.to_csv(
    PROCESSED_DIR / "X_train_encoded.csv",
    index=False
)

X_test_encoded.to_csv(
    PROCESSED_DIR / "X_test_encoded.csv",
    index=False
)

X_train_scaled.to_csv(
    PROCESSED_DIR / "X_train_scaled.csv",
    index=False
)

X_test_scaled.to_csv(
    PROCESSED_DIR / "X_test_scaled.csv",
    index=False
)

# Save training and testing targets
y_train.rename(TARGET).to_csv(
    PROCESSED_DIR / "y_train.csv",
    index=False
)

y_test.rename(TARGET).to_csv(
    PROCESSED_DIR / "y_test.csv",
    index=False
)

# Save fitted preprocessing objects
joblib.dump(
    encoder,
    MODEL_DIR / "one_hot_encoder.joblib"
)

joblib.dump(
    scaler,
    MODEL_DIR / "standard_scaler.joblib"
)

# Save feature metadata
pd.Series(
    numerical_features,
    name="Numerical_Feature"
).to_csv(
    PROCESSED_DIR / "numerical_features.csv",
    index=False
)

pd.Series(
    X_train_encoded.columns,
    name="Processed_Feature"
).to_csv(
    PROCESSED_DIR / "processed_feature_names.csv",
    index=False
)

# Save evidence from the leakage investigation
leakage_comparison.to_csv(
    PROCESSED_DIR / "target_leakage_comparison.csv",
    index=False
)

print("Processed datasets and preprocessing objects saved successfully.")

Processed datasets and preprocessing objects saved successfully.


## Verify Saved Files

In [92]:
for file_path in sorted(PROCESSED_DIR.glob("*")):
    print(file_path.name)

for file_path in sorted(MODEL_DIR.glob("*.joblib")):
    print(file_path.name)

numerical_features.csv
processed_feature_names.csv
target_leakage_comparison.csv
X_encoded.csv
X_scaled.csv
X_test_encoded.csv
X_test_scaled.csv
X_train_encoded.csv
X_train_scaled.csv
y.csv
y_test.csv
y_train.csv
one_hot_encoder.joblib
standard_scaler.joblib


# 19. Key Preprocessing Findings

The preprocessing stage produced the following key findings and methodological outcomes:

- The original dataset contained 100,000 observations and 20 variables, with no missing values.
- `Employee_ID` was removed because it is a unique identifier with no explanatory value.
- `Hire_Date` was removed because it contained one invalid constant value and therefore had no predictive value.
- A diagnostic modelling experiment compared performance before and after the removal of `Monthly_Salary`, `Promotions` and `Resigned`.
- The initial feature set produced a Decision Tree accuracy of 1.0000. Removing Monthly_Salary reduced accuracy to 0.1995, showing that salary was primarily responsible for the implausibly high initial performance.
- The subsequent removal of Promotions and Resigned resulted in comparatively small changes in accuracy, indicating that their exclusion was justified mainly by temporal ordering and conceptual ambiguity rather than by evidence that they independently caused the perfect initial result.
- Monthly_Salary, Promotions and Resigned were excluded from the final predictor set because they presented potential leakage, temporal-ordering or prospective-validity concerns.
- The final predictor dataset contains 14 original explanatory variables: 10 numerical and 4 categorical.
- The categorical variables were transformed using One-Hot Encoding fitted exclusively on the training data.
- The transformation produced 29 machine learning-ready predictor features.
- Numerical features were standardised using parameters learned only from the training data.
- Encoded datasets were retained for tree-based algorithms, while scaled datasets were created for scale-sensitive algorithms.
- An 80:20 stratified train/test split preserved the balanced distribution of the five employee performance categories.
- All transformed datasets passed checks for missing values, column consistency, numerical compatibility and target alignment.
- The processed datasets, fitted encoder, fitted scaler, feature names and target-leakage comparison were saved for reproducible use in subsequent notebooks.

Overall, this preprocessing stage demonstrates that high predictive accuracy is not, by itself, sufficient evidence of a valid machine learning model. Predictor validity must also be evaluated by considering when each variable becomes available and whether it could represent a consequence, proxy or indirect encoding of the target outcome.

The staged diagnostic experiment showed that Monthly_Salary was primarily responsible for the perfect initial Decision Tree accuracy. Although Promotions and Resigned did not independently produce a comparable increase in predictive performance, their temporal and conceptual status remained unsuitable for a prospective employee-performance prediction framework.

The final analytical datasets therefore prioritise methodological validity over artificially inflated predictive performance. They provide a transparent, leakage-controlled and reproducible foundation for the comparative model evaluation in Notebook 3 and the Explainable Artificial Intelligence analysis in Notebook 4.

# Transition to Machine Learning Model Development

The final encoded and scaled datasets are now ready for supervised machine learning.

Notebook 3 will:

- establish a majority-class baseline;
- evaluate Logistic Regression;
- evaluate Decision Tree;
- evaluate Random Forest;
- evaluate XGBoost;
- evaluate Linear Support Vector Machine;
- compare accuracy, precision, recall and weighted F1-score;
- validate the selected model using five-fold cross-validation; and
- select an appropriate model for SHAP-based explainability.

All final models will use the leakage-controlled feature set developed in this notebook.